# Wildflower Drought-Resistant Plants Ingestion

Read CSV plant lists from the `plant_chars` volume and ingest into a raw Delta table in `powers_land_catalog.natives`.

In [0]:
from pyspark.sql.functions import col, regexp_extract, url_decode

volume_path = "/Volumes/powers_land_catalog/natives/plant_chars/dought_resistant_plant_list - *.csv"

raw_df = (
    spark.read
    .option("header", "true")
    .csv(volume_path)
    .withColumn("_file_path", url_decode(col("_metadata.file_path")))
    .withColumn(
        "source_list",
        regexp_extract(
            col("_file_path"),
            r"dought_resistant_plant_list - (.+)\.csv$",
            1,
        ),
    )
    .drop("_file_path")
    .withColumnRenamed("Scientific Name", "scientific_name")
    .withColumnRenamed("Common Name", "common_name")
)

display(raw_df)

In [0]:
raw_df.write.mode("overwrite").saveAsTable(
    "powers_land_catalog.natives.raw_wildflower_org_plants"
)

In [0]:
%sql
SELECT source_list, count(*) AS row_count
FROM powers_land_catalog.natives.raw_wildflower_org_plants
GROUP BY source_list
ORDER BY source_list